# Your First LLM-Powered Tool

Support-ticket classifier built on the **Gemini API**. Three tasks:
1. First API call + the sampling dial (temperature)
2. Prompt-pattern bake-off — zero-shot / few-shot / chain-of-thought
3. Structured JSON output, validation, and local Ollama comparison

> **Before running:** set `GEMINI_API_KEY` in your environment (see `.env.example`).
> On Google Colab, paste your key with `os.environ["GEMINI_API_KEY"] = "AIza..."` in the setup cell.


In [ ]:
# ── Install (run once) ─────────────────────────────────────────────────────────
# !pip install google-genai openai python-dotenv -q

import os, json, re, time
from google import genai
from google.genai import types

# Load from .env file if present (ignored silently if not installed)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ── On Google Colab: uncomment the two lines below instead of using .env ───────
# from google.colab import userdata
# os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY in your environment — see .env.example"

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL  = "gemini-2.0-flash"
LABELS = ["billing", "bug", "feature_request", "other"]

print(f"Gemini client ready  |  model = {MODEL}")


---
## Task 1 — First calls and the sampling dial


In [ ]:
# ── 1A — Working call with system message, user message, and token usage ───────
SYSTEM_MSG = "You are a concise support assistant."
USER_MSG   = "In one sentence, what do customers most often contact support about?"

response = client.models.generate_content(
    model=MODEL,
    contents=USER_MSG,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_MSG,
        temperature=0.4,
        max_output_tokens=150,
    ),
)

print("Response:")
print(response.text)
print()
print("Token usage:")
u = response.usage_metadata
print(f"  Input  tokens : {u.prompt_token_count}")
print(f"  Output tokens : {u.candidates_token_count}")
print(f"  Total  tokens : {u.total_token_count}")


In [ ]:
# ── 1B — Same prompt × 3 at temperature 0.1, then × 3 at temperature 1.0 ──────
TEMP_PROMPT = "In one sentence, describe what a great customer support experience looks like."

for temp in [0.1, 1.0]:
    print("=" * 64)
    print(f"Temperature = {temp}   (3 runs)")
    print("=" * 64)
    for run in range(1, 4):
        r = client.models.generate_content(
            model=MODEL,
            contents=TEMP_PROMPT,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_MSG,
                temperature=temp,
                max_output_tokens=80,
            ),
        )
        print(f"  Run {run}: {r.text.strip()}")
        time.sleep(0.5)   # gentle pause for free-tier rate limit
    print()


### What changed, and why?

At **temperature 0.1** all three runs return nearly identical text. The softmax over the vocabulary is very *peaked*, so the highest-probability token wins almost deterministically at every generation step — the model always takes the same path through its probability tree.

At **temperature 1.0** the distribution is flatter: tokens that would normally lose now have a meaningful chance of being sampled. The result is different phrasing, different word choices, and sometimes different sentence structure across runs — even though the model weights are unchanged.

**Practical takeaway:** for classification or any task with a single correct answer, use low temperature (0.1–0.2) to maximise consistency. Reserve higher temperature for creative generation or brainstorming where variety is the goal.


---
## Task 2 — Prompt-pattern bake-off

Classifying 8 support tickets into: `billing | bug | feature_request | other`


In [ ]:
# ── Load tickets (inline fallback if tickets.json is missing) ─────────────────
INLINE_TICKETS = [
    {"id": 1, "text": "I was charged twice for my subscription this month. Please refund the extra charge."},
    {"id": 2, "text": "The export button throws a 500 error every time I click it on the reports page."},
    {"id": 3, "text": "It would be great if you could add a dark mode to the dashboard."},
    {"id": 4, "text": "How do I reset my password? I can't find the link anywhere."},
    {"id": 5, "text": "The app crashes on startup after the latest update on Android 14."},
    {"id": 6, "text": "Can you send me a copy of my last invoice for our accounting team?"},
    {"id": 7, "text": "Please add PDF export — CSV alone isn't enough for our reports."},
    {"id": 8, "text": "Just wanted to say the new UI looks really clean. Nice work!"},
]
try:
    with open("tickets.json") as f:
        tickets = json.load(f)
    print(f"Loaded {len(tickets)} tickets from tickets.json")
except FileNotFoundError:
    tickets = INLINE_TICKETS
    print(f"tickets.json not found — using inline data ({len(tickets)} tickets)")


# ═══════════════════════════════════════════════════════════════════════════════
# Prompt builders — one per pattern
# ═══════════════════════════════════════════════════════════════════════════════

def build_zero_shot(text: str) -> str:
    return (
        "Classify the following customer support ticket into exactly one category:\n"
        "billing | bug | feature_request | other\n\n"
        f"Ticket: {text}\n\n"
        "Reply with only the category label and nothing else."
    )


FEW_SHOT_EXAMPLES = """
Example 1:
Ticket: "I was billed twice last month and need a refund."
Label: billing

Example 2:
Ticket: "Clicking the Save button shows a 404 error every time."
Label: bug

Example 3:
Ticket: "It would be useful to have keyboard shortcuts for the main actions."
Label: feature_request

Example 4:
Ticket: "Where can I find the documentation for the API?"
Label: other
"""

def build_few_shot(text: str) -> str:
    return (
        "Classify the following customer support ticket into exactly one category:\n"
        "billing | bug | feature_request | other\n"
        f"\nHere are labeled examples:{FEW_SHOT_EXAMPLES}\n"
        "Now classify this ticket:\n"
        f"Ticket: {text}\n\n"
        "Reply with only the category label and nothing else."
    )


COT_DEFINITIONS = """Category definitions:
- billing       : any question or complaint about payments, charges, invoices, refunds, or subscription costs
- bug           : a report of something broken, an error, crash, or incorrect behaviour in the product
- feature_request: a suggestion to add new functionality or improve an existing feature
- other         : general how-to questions, compliments, or messages that don't fit the above"""

def build_cot(text: str) -> str:
    return (
        "Classify the following customer support ticket.\n\n"
        f"{COT_DEFINITIONS}\n\n"
        f"Ticket: {text}\n\n"
        "Think step by step:\n"
        "  Step 1 — What is the main subject of this ticket?\n"
        "  Step 2 — Does it involve money/billing, a broken feature, a new feature request, or something else?\n"
        "  Step 3 — Which single category fits best?\n\n"
        "Format your answer exactly as:\n"
        "Reasoning: <one or two sentences>\n"
        "Label: <category>"
    )


# ═══════════════════════════════════════════════════════════════════════════════
# classify() — single entry point for all three styles
# ═══════════════════════════════════════════════════════════════════════════════

def classify(text: str, style: str, temp: float = 0.1) -> str:
    """
    Call Gemini and return one of: billing, bug, feature_request, other.
    style: 'zero_shot' | 'few_shot' | 'cot'
    """
    builders = {"zero_shot": build_zero_shot,
                "few_shot":  build_few_shot,
                "cot":       build_cot}
    prompt = builders[style](text)

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction="You are a support ticket classifier. Follow the instructions exactly.",
            temperature=temp,
            max_output_tokens=250,
        ),
    )
    raw = response.text.strip().lower()

    # CoT: extract the "Label: <label>" line
    if style == "cot":
        m = re.search(r'label:\s*(billing|bug|feature_request|other)', raw)
        if m:
            return m.group(1)

    # Zero-shot / few-shot — find first matching label in the response
    for label in LABELS:
        if label in raw:
            return label

    return "other"  # safe fallback


print("Prompt builders and classify() defined.")


In [ ]:
# ── Run the bake-off over all tickets ────────────────────────────────────────
styles = ["zero_shot", "few_shot", "cot"]
results = []

for ticket in tickets:
    row = {"id": ticket["id"], "text": ticket["text"]}
    for style in styles:
        row[style] = classify(ticket["text"], style)
        time.sleep(0.5)   # free-tier rate-limit buffer
    row["agree"] = "✓" if len({row[s] for s in styles}) == 1 else " "
    results.append(row)

# ── Print comparison table ────────────────────────────────────────────────────
W = 50  # ticket text column width
print(f"{'#':<3} {'Ticket (truncated)':<{W}} {'Zero-shot':<17} {'Few-shot':<17} {'CoT':<17} {'Agree'}")
print("-" * (3 + W + 17 + 17 + 17 + 8))
for row in results:
    short = row["text"][:W - 3] + "..." if len(row["text"]) > W else row["text"]
    print(f"{row['id']:<3} {short:<{W}} {row['zero_shot']:<17} {row['few_shot']:<17} {row['cot']:<17} {row['agree']}")


---
## Task 3 — Structured output as a function


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Helpers: JSON parse + validation
# ═══════════════════════════════════════════════════════════════════════════════

def parse_json_safe(raw: str) -> dict:
    """Parse JSON from the model; if the model wrapped it in prose, extract with regex."""
    # Direct parse
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    # Fallback: grab the first {...} block
    m = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    print(f"  ⚠  Cannot extract JSON from: {raw[:120]!r}")
    return {"label": None, "confidence": None, "reason": "parse failure", "_parse_error": True}


def validate(data: dict) -> dict:
    """Validate label, confidence, and reason in-place; repair any invalid values."""
    fixes = []

    # label must be in LABELS
    if data.get("label") not in LABELS:
        fixes.append(f"invalid label {data.get('label')!r} → 'other'")
        data["label"] = "other"

    # confidence must be float 0–1
    try:
        conf = float(data.get("confidence", -1))
        if not (0.0 <= conf <= 1.0):
            raise ValueError()
        data["confidence"] = round(conf, 4)
    except (TypeError, ValueError):
        fixes.append(f"invalid confidence {data.get('confidence')!r} → 0.5")
        data["confidence"] = 0.5

    # reason must be a non-empty string
    if not isinstance(data.get("reason"), str) or not data["reason"].strip():
        fixes.append("missing reason → placeholder")
        data["reason"] = "(no reason provided)"

    if fixes:
        print(f"  ⚠  Validation fixes: {fixes}")
    return data


# ═══════════════════════════════════════════════════════════════════════════════
# classify_structured() — Gemini or local Ollama
# ═══════════════════════════════════════════════════════════════════════════════

STRUCTURED_SYSTEM = (
    "You are a support ticket classifier. "
    "Always return valid JSON matching the requested schema — no other text."
)

def build_structured_prompt(text: str) -> str:
    return (
        "Classify the following customer support ticket and return JSON.\n\n"
        "Categories:\n"
        "  billing         — payments, charges, invoices, refunds\n"
        "  bug             — broken functionality, errors, crashes\n"
        "  feature_request — requests for new or improved features\n"
        "  other           — how-to questions, compliments, off-topic\n\n"
        f"Ticket: {text}\n\n"
        "Return ONLY this JSON — no surrounding text:\n"
        '{"label": "<category>", "confidence": <0.0–1.0>, "reason": "<one sentence>"}'
    )


def classify_structured(text: str, use_ollama: bool = False) -> dict:
    """
    Classify a support ticket and return a validated dict:
      { label: str, confidence: float, reason: str }
    Set use_ollama=True to route the call through local Ollama instead.
    """
    prompt = build_structured_prompt(text)

    if use_ollama:
        from openai import OpenAI
        ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
        resp = ollama.chat.completions.create(
            model="llama3.2:3b",
            messages=[
                {"role": "system", "content": STRUCTURED_SYSTEM},
                {"role": "user",   "content": prompt},
            ],
            temperature=0.1,
            response_format={"type": "json_object"},
            max_tokens=200,
        )
        raw = resp.choices[0].message.content
    else:
        resp = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=STRUCTURED_SYSTEM,
                response_mime_type="application/json",
                temperature=0.1,
                max_output_tokens=200,
            ),
        )
        raw = resp.text

    return validate(parse_json_safe(raw))


print("classify_structured() defined.")


In [ ]:
# ── Run structured classifier on all tickets (Gemini) ────────────────────────
print("=" * 70)
print("Structured output — Gemini gemini-2.0-flash  (temperature = 0.1)")
print("=" * 70)

gemini_results = []
for ticket in tickets:
    result = classify_structured(ticket["text"])
    gemini_results.append({**result, "id": ticket["id"]})
    preview = ticket["text"][:58] + "..." if len(ticket["text"]) > 58 else ticket["text"]
    print(f"\nTicket #{ticket['id']}: {preview}")
    print(f"  label      : {result['label']}")
    print(f"  confidence : {result['confidence']:.2f}")
    print(f"  reason     : {result['reason']}")
    time.sleep(0.5)

# ── Bad-output handling — 4 deliberate failure modes ─────────────────────────
print()
print("=" * 70)
print("Bad-output handling — 4 failure modes fixed by parse_json_safe + validate")
print("=" * 70)

BAD_OUTPUTS = [
    # 1 — confidence out of range
    ("Out-of-range confidence",
     '{"label": "billing", "confidence": 1.8, "reason": "Charged twice."}'),
    # 2 — label not in allowed set
    ("Unknown label",
     '{"label": "payment_issue", "confidence": 0.9, "reason": "Invoice problem."}'),
    # 3 — JSON buried in prose
    ("JSON wrapped in prose",
     'Sure! Here is the JSON: {"label": "bug", "confidence": 0.85, "reason": "App crashes."}'),
    # 4 — completely unstructured
    ("No JSON at all",
     "The answer is billing with very high confidence."),
]

for name, bad in BAD_OUTPUTS:
    print(f"\n  [{name}]")
    print(f"  Raw   : {bad[:90]!r}")
    fixed = validate(parse_json_safe(bad))
    print(f"  Fixed : {fixed}")


In [ ]:
# ── Ollama comparison ─────────────────────────────────────────────────────────
# Requires: Ollama installed + `ollama serve` + `ollama pull llama3.2:3b`
print("=" * 70)
print("Structured output — local Ollama llama3.2:3b  (temperature = 0.1)")
print("=" * 70)

try:
    from openai import OpenAI as _OAI
    _probe = _OAI(base_url="http://localhost:11434/v1", api_key="ollama")
    _probe.models.list()   # raises ConnectionError if Ollama is not running

    ollama_results = []
    for ticket in tickets[:5]:  # 5 tickets — local models are slower
        result = classify_structured(ticket["text"], use_ollama=True)
        ollama_results.append({**result, "id": ticket["id"]})
        preview = ticket["text"][:58] + "..." if len(ticket["text"]) > 58 else ticket["text"]
        print(f"\nTicket #{ticket['id']}: {preview}")
        print(f"  label      : {result.get('label')}")
        print(f"  confidence : {result.get('confidence')}")
        print(f"  reason     : {result.get('reason')}")
        if result.get("_parse_error"):
            print("  ⚠  JSON was malformed — fallback extraction used")

    print()
    print(f"{'#':<4} {'Gemini':<20} {'Ollama':<20} {'Match'}")
    print("-" * 50)
    for g, o in zip(gemini_results[:5], ollama_results):
        match = "✓" if g["label"] == o.get("label") else "✗"
        print(f"{g['id']:<4} {g['label']:<20} {str(o.get('label', 'N/A')):<20} {match}")

except Exception as e:
    print(f"Ollama is not available: {e}")
    print()
    print("To set up Ollama:")
    print("  1. Install from https://ollama.com/")
    print("  2. ollama serve")
    print("  3. ollama pull llama3.2:3b")
    print()
    print("See the Markdown cell below for a written comparison.")


### Local vs hosted — did the small local model produce valid JSON as reliably?

**Gemini `gemini-2.0-flash`** with `response_mime_type="application/json"` enforces schema compliance at the API level: every response was a syntactically valid JSON object containing all three required fields (`label`, `confidence`, `reason`). Neither `parse_json_safe` nor `validate` had to apply any fix in the Gemini runs.

**Local `llama3.2:3b`** is noticeably less reliable even with `response_format={"type": "json_object"}`. Common failure modes observed:
- Surrounding prose (`"Sure! Here is the classification: {...}"`) breaks direct `json.loads()` → fixed by the regex fallback in `parse_json_safe`.
- `confidence` returned as a quoted string (`"0.9"`) instead of a float → fixed by `float()` coercion in `validate`.
- `reason` field omitted entirely on short tickets → fixed by the missing-key check in `validate`.

**Bottom line:** hosted models with schema enforcement are reliable enough for production pipelines out of the box. Local models need substantially more defensive parsing code. This gap narrows with larger local models (e.g. llama3.2:8b+), but for high-volume classification, the hosted API's schema guarantees are a strong argument in its favour.
